## Stacking Ensemble

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False

import warnings
warnings.filterwarnings('ignore')

df = pd.read_csv("../data/전처리완료.csv")

In [2]:
# 데이터 분리 라이브러리
from sklearn.model_selection import train_test_split

# 특징 변수(X)와 목표 변수(y) 분리
X = df.drop('고객이탈여부', axis=1)
y = df['고객이탈여부']

# 학습 데이터와 테스트 데이터 분리
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [3]:
# 필요한 라이브러리 불러오기
from sklearn.ensemble import StackingClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import train_test_split


In [4]:
# 기본 학습기 정의 (Base Learners)
base_learners = [
    ('rf', RandomForestClassifier(n_estimators=10, random_state=42)),
    ('gbm', GradientBoostingClassifier(n_estimators=100, learning_rate=0.1, max_depth=3, random_state=42))
]

# 메타 모델 정의 (Logistic Regression 사용)
meta_model = LogisticRegression()

# 스태킹 앙상블 모델 구성
stacking_clf = StackingClassifier(estimators=base_learners, final_estimator=meta_model)

# 모델 학습
stacking_clf.fit(X_train, y_train)

# 테스트 데이터로 예측
y_pred = stacking_clf.predict(X_test)

In [5]:
# 성과평가 라이브러리 불러오기
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from sklearn.metrics import f1_score, precision_score, recall_score
from sklearn.metrics import roc_curve, roc_auc_score

# 평가
print("Accuracy:", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall", recall_score(y_test, y_pred))
print("f1_score", f1_score(y_test, y_pred))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))
print("Classification Report:\n", classification_report(y_test, y_pred))

Accuracy: 0.8705
Precision: 0.7658730158730159
Recall 0.4910941475826972
f1_score 0.5984496124031008
Confusion Matrix:
 [[1548   59]
 [ 200  193]]
Classification Report:
               precision    recall  f1-score   support

           0       0.89      0.96      0.92      1607
           1       0.77      0.49      0.60       393

    accuracy                           0.87      2000
   macro avg       0.83      0.73      0.76      2000
weighted avg       0.86      0.87      0.86      2000



In [6]:
from sklearn.model_selection import cross_val_score

meta_scores = cross_val_score(stacking_clf, X_train, y_train, cv=5)
print(f"Stacking Meta-Model CV score: {meta_scores.mean():.4f}")

Stacking Meta-Model CV score: 0.8590
